# Sampling

We want to collect reviews wrote by active users, meaning :
- Users with more than 20 reviews
- A sample of 50,000 users among them 
- For active each user, collect all his reviews

### First things first ###
Convert the JSONL to Parquet first. Every subsequent read will be 10-50x faster:

In [1]:
import polars as pl
import duckdb

# With Polars:
pl.scan_ndjson("data/Books.jsonl").sink_parquet("data/Books-polars.parquet")

# # With DuckDB:
# duckdb.sql("""
#     COPY (SELECT * FROM read_json_auto('data/Books.jsonl', format='newline_delimited'))
#     TO 'data/Books.parquet' (FORMAT PARQUET, ROW_GROUP_SIZE 1000000)
# """)

# DuckDB handles messy JSONL flawlessly and writes Parquet very fast
con = duckdb.connect()
print("Converting JSONL → Parquet (this takes ~5-10 min)...")
con.execute("""
    COPY (
        SELECT *
        FROM read_json_auto(
            'data/Books.jsonl',
            format='newline_delimited',
            maximum_object_size=10485760
        )
    ) TO 'data/Books-duckdb.parquet' (FORMAT PARQUET, ROW_GROUP_SIZE 1000000)
""")
print("Done!")
con.close()

Converting JSONL → Parquet (this takes ~5-10 min)...
Done!


### First Sampling iteration Streaming Python ###

The simplest and most memory-efficient. Uses almost zero RAM beyond the dictionaries.


In [9]:
import json
import random
from collections import Counter
from datetime import datetime

DATA_PATH = "data/Books.jsonl"
OUTPUT_PATH = "sample-streaming-python/sampled_reviews.jsonl"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ── Pass 1: Count reviews per user ──────────────────────────────
print("Pass 1: Counting reviews per user...")
user_counts = Counter()

with open(DATA_PATH, "r") as f:
    for i, line in enumerate(f):
        record = json.loads(line)
        user_counts[record["user_id"]] += 1
        if i % 5_000_000 == 0:
            print(f"  Processed {i:,} lines...")

print(f"Total unique users: {len(user_counts):,}")

# ── Filter active users (>= 20 reviews) ────────────────────────
active_users = [uid for uid, count in user_counts.items() if count >= MIN_REVIEWS]
print(f"Active users (>= {MIN_REVIEWS} reviews): {len(active_users):,}")

# ── Sample 50,000 users ────────────────────────────────────────
random.seed(SEED)
sampled_users = set(random.sample(active_users, min(NUM_USERS, len(active_users))))
print(f"Sampled users: {len(sampled_users):,}")

del user_counts, active_users  # free memory

# ── Pass 2: Extract reviews for sampled users ──────────────────
print("Pass 2: Extracting reviews for sampled users...")
total_extracted = 0

with open(DATA_PATH, "r") as fin, open(OUTPUT_PATH, "w") as fout:
    for i, line in enumerate(fin):
        record = json.loads(line)
        if record["user_id"] in sampled_users:
            fout.write(line)
            total_extracted += 1
        if i % 5_000_000 == 0:
            print(f"  Processed {i:,} lines, extracted {total_extracted:,}")

print(f"Total extracted reviews: {total_extracted:,}")
    


Pass 1: Counting reviews per user...
  Processed 0 lines...
  Processed 5,000,000 lines...
  Processed 10,000,000 lines...
  Processed 15,000,000 lines...
  Processed 20,000,000 lines...
  Processed 25,000,000 lines...
Total unique users: 10,297,355
Active users (>= 20 reviews): 137,305
Sampled users: 50,000
Pass 2: Extracting reviews for sampled users...
  Processed 0 lines, extracted 0
  Processed 5,000,000 lines, extracted 745,349
  Processed 10,000,000 lines, extracted 1,323,541
  Processed 15,000,000 lines, extracted 1,852,537
  Processed 20,000,000 lines, extracted 2,204,841
  Processed 25,000,000 lines, extracted 2,428,722
Total extracted reviews: 2,442,098


### Second Sampling iteration Using Pandas with Chunked Reading ###

In [10]:
import pandas as pd
import random
from collections import Counter

DATA_PATH = "data/Books.jsonl"
CHUNK_SIZE = 500_000  # rows per chunk
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ── Pass 1: Count reviews per user (chunked) ───────────────────
print("Pass 1: Counting reviews per user...")
user_counts = Counter()

reader = pd.read_json(DATA_PATH, lines=True, chunksize=CHUNK_SIZE,
                       dtype={"user_id": str, "parent_asin": str, "rating": float})

for i, chunk in enumerate(reader):
    counts = chunk["user_id"].value_counts()
    user_counts.update(counts.to_dict())
    print(f"  Chunk {i}: {len(chunk):,} rows processed")

# ── Filter & sample active users ───────────────────────────────
active_users = [uid for uid, c in user_counts.items() if c >= MIN_REVIEWS]
print(f"Active users: {len(active_users):,}")

random.seed(SEED)
sampled_users = set(random.sample(active_users, min(NUM_USERS, len(active_users))))
del user_counts, active_users

# ── Pass 2: Collect reviews for sampled users ──────────────────
print("Pass 2: Extracting reviews...")
chunks = []
reader = pd.read_json(DATA_PATH, lines=True, chunksize=CHUNK_SIZE,
                       dtype={"user_id": str, "parent_asin": str, "rating": float})

for i, chunk in enumerate(reader):
    filtered = chunk[chunk["user_id"].isin(sampled_users)]
    if len(filtered) > 0:
        chunks.append(filtered)
    print(f"  Chunk {i}: kept {len(filtered):,} / {len(chunk):,}")

df = pd.concat(chunks, ignore_index=True)
print(f"Final sample: {len(df):,} reviews from {df['user_id'].nunique():,} users")

# ── Save ────────────────────────────────────────────────────────
df.to_parquet("sample-pandas-with-chunk/sampled_reviews.parquet", index=False)
df.to_json("sample-pandas-with-chunk/sampled_reviews.jsonl", orient="records", lines=True)

Pass 1: Counting reviews per user...
  Chunk 0: 500,000 rows processed
  Chunk 1: 500,000 rows processed
  Chunk 2: 500,000 rows processed
  Chunk 3: 500,000 rows processed
  Chunk 4: 500,000 rows processed
  Chunk 5: 500,000 rows processed
  Chunk 6: 500,000 rows processed
  Chunk 7: 500,000 rows processed
  Chunk 8: 500,000 rows processed
  Chunk 9: 500,000 rows processed
  Chunk 10: 500,000 rows processed
  Chunk 11: 500,000 rows processed
  Chunk 12: 500,000 rows processed
  Chunk 13: 500,000 rows processed
  Chunk 14: 500,000 rows processed
  Chunk 15: 500,000 rows processed
  Chunk 16: 500,000 rows processed
  Chunk 17: 500,000 rows processed
  Chunk 18: 500,000 rows processed
  Chunk 19: 500,000 rows processed
  Chunk 20: 500,000 rows processed
  Chunk 21: 500,000 rows processed
  Chunk 22: 500,000 rows processed
  Chunk 23: 500,000 rows processed
  Chunk 24: 500,000 rows processed
  Chunk 25: 500,000 rows processed
  Chunk 26: 500,000 rows processed
  Chunk 27: 500,000 rows pro

### Third Sampling iteration Using Polars ###

Polars is a Rust-based DataFrame library that is 5-10x faster than pandas for this workload thanks to multi-threaded execution and lazy evaluation.

In [13]:
import polars as pl
import random

DATA_PATH = "data/Books.jsonl"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ── Polars lazy scan (memory-mapped, multi-threaded) ────────────
# This does NOT load the file into RAM — it creates a query plan
lf = pl.scan_ndjson(
    DATA_PATH,
    ignore_errors=True,
    low_memory=False,
    # Don't specify full schema — let Polars infer with overrides
    schema_overrides={
        "user_id": pl.Utf8,
        "rating": pl.Float64,
        "timestamp": pl.Int64,
    },
)

# ── Step 1: Find active users ──────────────────────────────────
print("Step 1: Identifying active users...")
user_review_counts = (
    lf.select("user_id")           # <── project FIRST, reduces data + schema issues
    .group_by("user_id")
    .agg(pl.len().alias("review_count"))
    .filter(pl.col("review_count") >= MIN_REVIEWS)
    .collect()  # materializes only the aggregation result
)
print(f"Active users (>= {MIN_REVIEWS}): {len(user_review_counts):,}")

# ── Step 2: Sample 50,000 users ────────────────────────────────
random.seed(SEED)
all_active = user_review_counts["user_id"].to_list()
sampled_users = random.sample(all_active, min(NUM_USERS, len(all_active)))
sampled_set = pl.Series("user_id", sampled_users)

# ── Step 3: Filter reviews ─────────────────────────────────────
print("Step 2: Filtering reviews for sampled users...")
sampled_df = (
    lf.filter(pl.col("user_id").is_in(sampled_set))
    .collect()
)
print(f"Sampled: {len(sampled_df):,} reviews, {sampled_df['user_id'].n_unique():,} users")

# ── Optional temporal filter (combine strategies) ───────────────
# Unix timestamp for Jan 1, 2020 = 1577836800
# Unix timestamp for Dec 31, 2023 = 1703980800
sampled_temporal = sampled_df.filter(
    (pl.col("timestamp") >= 1577836800) & (pl.col("timestamp") <= 1703980800)
)
print(f"After temporal filter (2020-2023): {len(sampled_temporal):,} reviews")

# ── Save ────────────────────────────────────────────────────────
sampled_df.write_parquet("sample-polars/sampled_reviews.parquet")
sampled_df.write_json("sample-polars/sampled_reviews.jsonl")

Step 1: Identifying active users...
Active users (>= 20): 137,305
Step 2: Filtering reviews for sampled users...


/tmp/ipykernel_3232/3053399840.py:44: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .collect()


Sampled: 2,425,431 reviews, 50,000 users
After temporal filter (2020-2023): 0 reviews


### Fourth Sampling iteration using Dask ###

Dask extends pandas to larger-than-memory datasets with lazy parallel execution.

In [ ]:
import dask.dataframe as dd
from dask.distributed import Client, LocalCluster
import random

DATA_PATH = "data/Books-polars.parquet"  # USE PARQUET, not JSONL!
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ── Configure Dask with explicit memory limits ─────────────────
# This prevents workers from consuming all RAM and crashing the machine
cluster = LocalCluster(
    n_workers=4,              # fewer workers = less concurrent memory
    threads_per_worker=6,
    memory_limit="12GB",      # per-worker limit (4 × 12 = 48GB ceiling)
)
client = Client(cluster)
print(client.dashboard_link)  # monitor at http://localhost:8787

# ── Step 1: Count reviews per user (project ONLY user_id) ──────
print("Counting reviews per user...")
ddf = dd.read_parquet(DATA_PATH, columns=["user_id"])  # only 1 column!
user_counts = ddf.groupby("user_id").size().compute()  # small Series result

active_users = user_counts[user_counts >= MIN_REVIEWS].index.tolist()
print(f"Active users: {len(active_users):,}")

del ddf, user_counts  # free Dask graph references

# ── Step 2: Sample users (CPU, trivial) ────────────────────────
random.seed(SEED)
sampled_users = set(random.sample(active_users, min(NUM_USERS, len(active_users))))
del active_users

# ── Step 3: Filter reviews and write directly to disk ──────────
print("Filtering reviews...")
NEEDED_COLS = ["user_id", "parent_asin", "rating", "timestamp",
               "title", "text", "helpful_vote", "verified_purchase", "asin"]
ddf_full = dd.read_parquet(DATA_PATH, columns=NEEDED_COLS)
sampled_ddf = ddf_full[ddf_full["user_id"].isin(sampled_users)]

# Write directly to parquet WITHOUT calling .compute()
# This streams partitions to disk one at a time instead of materializing all in RAM
sampled_ddf.to_parquet(
    "sample-dask/sampled_reviews-polars.parquet",
    write_index=False,
    overwrite=True,
)

# If you need a single file or want to also get the count:
result = dd.read_parquet("sample-dask/sampled_reviews-polars.parquet")
print(f"Sampled: {len(result):,} reviews")

# ── Cleanup ────────────────────────────────────────────────────
client.close()
cluster.close()

import pandas as pd

# Read back the already-filtered, much smaller parquet
sampled_df = pd.read_parquet("sample-dask/sampled_reviews-polars.parquet")
print(f"Sampled: {len(sampled_df):,} reviews from {sampled_df['user_id'].nunique():,} users")
sampled_df.to_json("sample-dask/sampled_reviews-polars.jsonl", orient="records", lines=True)

/usr/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 37673 instead
  warnings.warn(


http://127.0.0.1:37673/status
Counting reviews per user...


Active users: 137,305
Filtering reviews...
Sampled: 2,415,706 reviews
Sampled: 2,415,706 reviews from 50,000 users


### Fifth Sampling iteration using cuDF / RAPIDS ###

This is the GPU powerhouse. cuDF mirrors the pandas API but runs on NVIDIA GPUs. You need an NVIDIA GPU with sufficient VRAM (ideally 16GB+ for this dataset).

#### Option A ####

If the dataset fits in GPU VRAM (24GB+ GPU), cuDF can read JSONL directly on GPU.

In [9]:
import cudf
import pyarrow.parquet as pq
import pandas as pd
import random

DATA_PATH = "data/Books-polars.parquet"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ============================================================
# PHASE 1: Find active users and sample 50,000
# (Only loads the user_id column — very light on VRAM)
# ============================================================

print("Phase 1: Counting reviews per user...")
gdf_ids = cudf.read_parquet(DATA_PATH, columns=["user_id"])

user_counts = gdf_ids["user_id"].value_counts().reset_index()
user_counts.columns = ["user_id", "count"]
active = user_counts[user_counts["count"] >= MIN_REVIEWS]
print(f"Active users (>= {MIN_REVIEWS} reviews): {len(active):,}")

# Sample 50,000 users on CPU
random.seed(SEED)
active_list = active["user_id"].to_pandas().tolist()
sampled = random.sample(active_list, min(NUM_USERS, len(active_list)))
sampled_list = list(sampled)
print(f"Sampled users: {len(sampled_list):,}")

# Free GPU memory completely before Phase 2
del gdf_ids, user_counts, active

# ============================================================
# PHASE 2: Extract all reviews for sampled users
# (Reads one row group at a time to avoid VRAM overflow)
# ============================================================

print("\nPhase 2: Extracting reviews for sampled users...")

pf = pq.ParquetFile(DATA_PATH)
n_row_groups = pf.metadata.num_row_groups
print(f"Parquet has {n_row_groups} row groups to process")

result_chunks = []
total_matched = 0

for rg in range(n_row_groups):
    # Load one row group into GPU (~1M rows, ~1-2 GB VRAM)
    chunk = cudf.read_parquet(DATA_PATH, row_groups=[rg])
    
    # Filter on GPU: keep only rows matching our 50K users
    filtered = chunk[chunk["user_id"].isin(sampled_list)]
    
    matched = len(filtered)
    total_matched += matched
    
    # Move small filtered result to CPU, free GPU memory
    if matched > 0:
        result_chunks.append(filtered.to_pandas())
    
    del chunk, filtered
    print(f"  Row group {rg + 1}/{n_row_groups}: matched {matched:,} rows (total: {total_matched:,})")

# ============================================================
# PHASE 3: Combine results and save
# ============================================================

print("\nPhase 3: Combining and saving...")
df = pd.concat(result_chunks, ignore_index=True)
print(f"Final sample: {len(df):,} reviews from {df['user_id'].nunique():,} users")

# Optional temporal filter (2020-2023)
df_temporal = df[(df["timestamp"] >= 1577836800) & (df["timestamp"] <= 1703980800)]
print(f"After temporal filter (2020-2023): {len(df_temporal):,} reviews")

# Save
df.to_parquet("sample-cudf-rapids-24+/sampled_reviews.parquet", index=False)
print("Saved to sample-cudf-rapids-24+/sampled_reviews.parquet")
df.to_json("sample-cudf-rapids-24+/sampled_reviews.jsonl",orient="records", lines=True)
print("Saved to sample-cudf-rapids-24+/sampled_reviews.jsonl")

Phase 1: Counting reviews per user...
Active users (>= 20 reviews): 137,305
Sampled users: 50,000

Phase 2: Extracting reviews for sampled users...
Parquet has 240 row groups to process
  Row group 1/240: matched 30,481 rows (total: 30,481)
  Row group 2/240: matched 26,172 rows (total: 56,653)
  Row group 3/240: matched 23,162 rows (total: 79,815)
  Row group 4/240: matched 16,972 rows (total: 96,787)
  Row group 5/240: matched 18,092 rows (total: 114,879)
  Row group 6/240: matched 22,846 rows (total: 137,725)
  Row group 7/240: matched 18,682 rows (total: 156,407)
  Row group 8/240: matched 14,575 rows (total: 170,982)
  Row group 9/240: matched 13,767 rows (total: 184,749)
  Row group 10/240: matched 14,876 rows (total: 199,625)
  Row group 11/240: matched 13,893 rows (total: 213,518)
  Row group 12/240: matched 14,344 rows (total: 227,862)
  Row group 13/240: matched 14,252 rows (total: 242,114)
  Row group 14/240: matched 17,361 rows (total: 259,475)
  Row group 15/240: matched 2

#### Option B ####

Chunked GPU processing is more memory-efficient for limited VRAM.

In [11]:
import cudf
import pandas as pd
import random
from collections import Counter

DATA_PATH = "data/Books.jsonl"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ── Pass 1: Count reviews per user (chunked on GPU) ────────────
print("Pass 1: Counting reviews per user on GPU...")
user_counts = Counter()

# Use byte_range to read the file in chunks
import os
file_size = os.path.getsize(DATA_PATH)
CHUNK_BYTES = 400_000_000_000  # 500MB chunks — adjust based on your VRAM

offset = 0
chunk_num = 0
while offset < file_size:
    size = min(CHUNK_BYTES, file_size - offset)
    try:
        chunk = cudf.read_json(
            DATA_PATH,
            lines=True,
            engine="cudf",
            byte_range=(offset, size),  # <── read only a slice of the file
        )
        # Aggregate on GPU, transfer tiny result to CPU
        counts = chunk["user_id"].value_counts()
        cpu_counts = counts.to_pandas()
        user_counts.update(cpu_counts.to_dict())
        
        del chunk, counts, cpu_counts  # free GPU memory immediately
        print(f"  Chunk {chunk_num}: offset={offset:,}, size={size:,}")
    except Exception as e:
        print(f"  Chunk {chunk_num} error (skipping): {e}")
    
    offset += size
    chunk_num += 1

# ── Filter active users ────────────────────────────────────────
active_users = [u for u, c in user_counts.items() if c >= MIN_REVIEWS]
print(f"Active users: {len(active_users):,}")

random.seed(SEED)
sampled_users = set(random.sample(active_users, min(NUM_USERS, len(active_users))))
del user_counts, active_users

# ── Pass 2: Filter reviews (chunked on GPU) ────────────────────
print("Pass 2: Extracting reviews on GPU...")
sampled_list = list(sampled_users)  # cudf.isin needs a list
result_chunks = []

offset = 0
while offset < file_size:
    size = min(CHUNK_BYTES, file_size - offset)
    try:
        chunk = cudf.read_json(
            DATA_PATH,
            lines=True,
            engine="cudf",
            byte_range=(offset, size),
        )
        # Filter on GPU, transfer only matching rows to CPU
        filtered = chunk[chunk["user_id"].isin(sampled_list)]
        if len(filtered) > 0:
            result_chunks.append(filtered.to_pandas())
        del chunk, filtered
    except Exception:
        pass

    offset += size

df = pd.concat(result_chunks, ignore_index=True)
print(f"Sampled: {len(df):,} reviews from {df['user_id'].nunique():,} users")

df.to_parquet("sample-cudf-rapids-24-/sampled_reviews.parquet", index=False)
df.to_json("sample-cudf-rapids-24-/sampled_reviews.jsonl", orient="records", lines=True)

Pass 1: Counting reviews per user on GPU...
  Chunk 0: offset=0, size=20,121,186,727
Active users: 137,305
Pass 2: Extracting reviews on GPU...
Sampled: 2,442,267 reviews from 50,000 users


#### Option C ####

In [2]:
import cudf
import cupy as cp
import rmm
import gc
import random
import time

DATA_PATH = "data/Books.jsonl"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ── Helper: flush everything ────────────────────────────────────
def flush_memory():
    """Aggressively free both RAM and VRAM."""
    # 1. Python garbage collector — release unreferenced objects
    gc.collect()
    
    # 2. CuPy memory pool — free cached GPU blocks
    mempool = cp.get_default_memory_pool()
    pinned_mempool = cp.get_default_pinned_memory_pool()
    mempool.free_all_blocks()
    pinned_mempool.free_all_blocks()
    
    # 3. Force CUDA synchronization (ensure all GPU ops finish first)
    cp.cuda.runtime.deviceSynchronize()

def print_memory_status(label=""):
    """Show current GPU memory usage."""
    import pynvml
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    print(f"  [{label}] GPU: {info.used/1e9:.2f}/{info.total/1e9:.2f} GB "
          f"(free: {info.free/1e9:.2f} GB)")
    pynvml.nvmlShutdown()


# ── Configure RMM memory pool ──────────────────────────────────
rmm.reinitialize(
    managed_memory=True,
    pool_allocator=True,
)

print_memory_status("Before start")

# ================================================================
# PHASE 1: Load JSONL, count per user, find active users
# ================================================================
start = time.time()
print("Phase 1: Chargement en GPU memory...")

gdf = cudf.read_json(DATA_PATH, lines=True)
gdf['rating'] = gdf['rating'].astype('int8')

print_memory_status("After load")
print(f"  GPU DataFrame: {gdf.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# Comptage sur GPU
user_counts = gdf['user_id'].value_counts()
active_users = user_counts[user_counts >= MIN_REVIEWS].index

# ── Transfer active user IDs to CPU immediately ────────────────
active_list = active_users.to_pandas().tolist()
print(f"  Active users: {len(active_list):,}")

# ── Sample 50,000 randomly (CPU) ───────────────────────────────
random.seed(SEED)
selected_users = random.sample(active_list, min(NUM_USERS, len(active_list)))

# ── FLUSH: delete the counts, we only need the user ID list now ─
del user_counts, active_users, active_list
flush_memory()
print_memory_status("After count flush")

# ================================================================
# PHASE 2: Filter the full DataFrame for sampled users
# ================================================================
print("\nPhase 2: Filtrage...")

# Convert back to cudf Series for GPU-side isin()
selected_series = cudf.Series(selected_users)
mask = gdf['user_id'].isin(selected_series)
sample_gdf = gdf[mask]

print(f"  Reviews matched: {len(sample_gdf):,}")

# ── FLUSH: delete the full DataFrame — we no longer need it ────
del gdf, mask, selected_series
flush_memory()
print_memory_status("After filter flush")

# ================================================================
# PHASE 3: Transfer to CPU and save
# ================================================================
print("\nPhase 3: Transfert vers CPU et sauvegarde...")

sample = sample_gdf.to_pandas()

# ── FLUSH: delete the GPU DataFrame — data is on CPU now ───────
del sample_gdf
flush_memory()
print_memory_status("After GPU->CPU flush")

elapsed = time.time() - start
print(f"\nTemps d'execution: {elapsed:.2f}s")
print(f"Reviews echantillonnees: {len(sample):,}")
print(f"Utilisateurs uniques: {sample['user_id'].nunique():,}")

# Save
sample.to_parquet('sample-cudf-claude/sample_gpu_active_users.parquet', compression='snappy')
sample.to_json("sample-cudf-claude/sample_gpu_active_users.jsonl", orient="records", lines=True)

# ── FINAL FLUSH: free everything including the pandas DataFrame ─
del sample
gc.collect()
print_memory_status("Final cleanup")

  [Before start] GPU: 34.18/34.19 GB (free: 0.01 GB)
Phase 1: Chargement en GPU memory...
  [After load] GPU: 33.84/34.19 GB (free: 0.35 GB)
  GPU DataFrame: 16.04 GB
  Active users: 137,305
  [After count flush] GPU: 33.83/34.19 GB (free: 0.36 GB)

Phase 2: Filtrage...
  Reviews matched: 2,442,267
  [After filter flush] GPU: 33.83/34.19 GB (free: 0.36 GB)

Phase 3: Transfert vers CPU et sauvegarde...
  [After GPU->CPU flush] GPU: 33.83/34.19 GB (free: 0.36 GB)

Temps d'execution: 10.03s
Reviews echantillonnees: 2,442,267
Utilisateurs uniques: 50,000
  [Final cleanup] GPU: 33.83/34.19 GB (free: 0.36 GB)


#### Option D ####

In [2]:
import cudf
import cupy as cp
import rmm


DATA_PATH = "data/Books.jsonl"
CHUNK_SIZE = 2_000_000
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# Configuration mémoire GPU
rmm.reinitialize(
    managed_memory=True,  # Unified memory CPU/GPU
    pool_allocator=True
)

def sample_with_gpu(DATA_PATH):
    """
    Échantillonnage GPU-accéléré avec cuDF (RAPIDS)
    
    Performance: 10-50x plus rapide que pandas
    Requis: GPU NVIDIA avec 8GB+ VRAM
    
    Installation:
    conda install -c rapidsai -c conda-forge -c nvidia \
        cudf cudatoolkit=13.1.1-1
    """
    
    # Lecture directe en GPU
    print("Chargement en GPU memory...")
    gdf = cudf.read_json(DATA_PATH, lines=True)
    
    # Optimisation types (GPU)
    gdf['rating'] = gdf['rating'].astype('int8')
    gdf['user_id'] = gdf['user_id'].astype('category')
    
    print(f"GPU memory utilisée: {gdf.memory_usage(deep=True).sum() / 1e9:.2f} GB")
    
    # Conversion timestamp
    gdf['timestamp'] = cudf.to_datetime(gdf['timestamp'])
    gdf['year'] = gdf['timestamp'].dt.year
    
    # Échantillonnage stratifié par année (GPU)
    samples = []
    for year in gdf['year'].unique().to_pandas():
        year_data = gdf[gdf['year'] == year]
        n_samples = min(200000, len(year_data))
        samples.append(year_data.sample(n=n_samples))
    
    sample_gpu = cudf.concat(samples)
    sample = sample_gpu.to_pandas()

    return sample

# Monitoring GPU
def monitor_gpu_memory():
    """Affiche utilisation GPU"""
    import pynvml
    
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    
    print(f"GPU Memory: {info.used/1e9:.2f}/{info.total/1e9:.2f} GB")
    pynvml.nvmlShutdown()

# Utilisation
if __name__ == '__main__':
    import time
    
    start = time.time()
    sample = sample_with_gpu(DATA_PATH)
    elapsed = time.time() - start
    
    print(f"\n⚡ Temps d'exécution GPU: {elapsed:.2f}s")
    print(f"   Reviews échantillonnées: {len(sample):,}")
    
    # Sauvegarde
    sample.to_parquet('sample-cudf-claude/sample_gpu_temporal.parquet', compression='snappy')
    sample.to_json("sample-cudf-claude/sample_gpu_temporal.jsonl", orient="records", lines=True)

Chargement en GPU memory...
GPU memory utilisée: 15.55 GB

⚡ Temps d'exécution GPU: 9.76s
   Reviews échantillonnées: 200,000


### Sixth Sampling iteration using Dask with cuDF ###

Combines Dask's out-of-core scheduling with cuDF's GPU execution.


In [ ]:
import dask_cudf
from dask_cuda import LocalCUDACluster
from dask.distributed import Client
import cudf
import random
import gc

DATA_PATH = "data/Books-polars.parquet"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

NEEDED_COLS = ["user_id", "parent_asin", "rating", "timestamp",
               "title", "text", "helpful_vote", "verified_purchase"]

# ── Configure a CUDA-aware Dask cluster with memory limits ─────
cluster = LocalCUDACluster(
    n_workers=1,                    # 1 GPU = 1 worker
    device_memory_limit="24GB",     # leave ~8GB VRAM headroom for CUDA overhead
    memory_limit="32GB",            # CPU RAM spill limit per worker
    jit_unspill=True,               # smart GPU<->CPU spilling
)
client = Client(cluster)
print(f"Dashboard: {client.dashboard_link}")

# ── Step 1: Count reviews per user (only user_id column) ───────
print("Counting reviews per user...")
ddf = dask_cudf.read_parquet(DATA_PATH, columns=["user_id"])
user_counts = ddf.groupby("user_id").size().compute()

active_users = user_counts[user_counts >= MIN_REVIEWS].index.to_arrow().to_pylist()
print(f"Active users: {len(active_users):,}")

del ddf, user_counts
gc.collect()

# ── Step 2: Sample users ──────────────────────────────────────
random.seed(SEED)
sampled_users = set(random.sample(active_users, min(NUM_USERS, len(active_users))))
del active_users

# ── Step 3: Filter and WRITE TO DISK (never .compute()) ───────
print("Filtering reviews...")
ddf_full = dask_cudf.read_parquet(DATA_PATH, columns=NEEDED_COLS)
sampled_ddf = ddf_full[ddf_full["user_id"].isin(list(sampled_users))]

# Write partition-by-partition to disk -- each partition is processed
# on GPU then written out, freeing VRAM before the next one
sampled_ddf.to_parquet(
    "sample-dask-cuda/sampled_reviews.parquet",
    write_index=False,
    overwrite=True,
)
print("Parquet written!")

# ── Cleanup GPU resources ─────────────────────────────────────
client.close()
cluster.close()
gc.collect()

# ── Now read back the small result on CPU ─────────────────────
import pandas as pd
sampled_df = pd.read_parquet("sample-dask-cuda/sampled_reviews.parquet")
print(f"Sampled: {len(sampled_df):,} reviews from {sampled_df['user_id'].nunique():,} users")
sampled_df.to_json("sample-dask-cuda/sampled_reviews.jsonl", orient="records", lines=True)

Counting reviews per user...
Active users: 137,305
Filtering reviews...


### Seventh Sampling iteration using PySpark ###

For when you have a cluster or want Spark's optimizer.


In [ ]:
'''
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import random

spark = SparkSession.builder \
    .appName("AmazonReviewsSampling") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

DATA_PATH = "data/Books.jsonl"

# ── Load ────────────────────────────────────────────────────────
df = spark.read.json(DATA_PATH)

# ── Active users ────────────────────────────────────────────────
user_counts = df.groupBy("user_id").count().filter(F.col("count") >= 20)

# Sample 50K users using Spark's built-in sampling
# Approximate: calculate fraction needed
total_active = user_counts.count()
fraction = min(50000 / total_active, 1.0)
sampled_users = user_counts.sample(False, fraction, seed=42).limit(50000)

# ── Filter ──────────────────────────────────────────────────────
sampled_df = df.join(sampled_users.select("user_id"), on="user_id", how="inner")

# ── Optional temporal filter ────────────────────────────────────
sampled_df = sampled_df.filter(
    (F.col("timestamp") >= 1577836800) & (F.col("timestamp") <= 1703980800)
)

# ── Save ────────────────────────────────────────────────────────
sampled_df.coalesce(1).write.mode("overwrite").parquet("data/sampled_reviews_spark")

spark.stop()
'''

'\nfrom pyspark.sql import SparkSession\nfrom pyspark.sql import functions as F\nimport random\n\nspark = SparkSession.builder     .appName("AmazonReviewsSampling")     .config("spark.driver.memory", "8g")     .config("spark.sql.shuffle.partitions", "200")     .getOrCreate()\n\nDATA_PATH = "data/Books.jsonl"\n\n# ── Load ────────────────────────────────────────────────────────\ndf = spark.read.json(DATA_PATH)\n\n# ── Active users ────────────────────────────────────────────────\nuser_counts = df.groupBy("user_id").count().filter(F.col("count") >= 20)\n\n# Sample 50K users using Spark\'s built-in sampling\n# Approximate: calculate fraction needed\ntotal_active = user_counts.count()\nfraction = min(50000 / total_active, 1.0)\nsampled_users = user_counts.sample(False, fraction, seed=42).limit(50000)\n\n# ── Filter ──────────────────────────────────────────────────────\nsampled_df = df.join(sampled_users.select("user_id"), on="user_id", how="inner")\n\n# ── Optional temporal filter ───────

### Eighth Sampling iteration using DuckDB ###

DuckDB is an in-process OLAP database -- think "SQLite for analytics." Extremely fast for this kind of aggregation.

In [ ]:
import duckdb
import random

DATA_PATH = "data/Books.jsonl"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

con = duckdb.connect()

# DuckDB reads JSONL natively and very efficiently
# ── Step 1: Find active users ──────────────────────────────────
print("Finding active users...")
active_users = con.execute(f"""
    SELECT user_id, COUNT(*) as cnt
    FROM read_json_auto('{DATA_PATH}', format='newline_delimited', maximum_object_size=10485760)
    GROUP BY user_id
    HAVING cnt >= {MIN_REVIEWS}
""").fetchdf()

print(f"Active users: {len(active_users):,}")

# ── Step 2: Sample users ───────────────────────────────────────
random.seed(SEED)
sampled = random.sample(active_users["user_id"].tolist(), 
                        min(NUM_USERS, len(active_users)))

# Register as a DuckDB table for efficient join
con.execute("CREATE TABLE sampled_users (user_id VARCHAR)")
con.executemany("INSERT INTO sampled_users VALUES (?)", [(u,) for u in sampled])

# ── Step 3: Extract reviews ────────────────────────────────────
print("Extracting reviews...")
con.execute(f"""
    COPY (
        SELECT r.*
        FROM read_json_auto('{DATA_PATH}', format='newline_delimited', 
                            maximum_object_size=10485760) r
        INNER JOIN sampled_users s ON r.user_id = s.user_id
    ) TO 'sample-duckdb/sampled_reviews.parquet' (FORMAT PARQUET)
""")

# ── With temporal filter ────────────────────────────────────────
con.execute(f"""
    COPY (
        SELECT r.*
        FROM read_json_auto('{DATA_PATH}', format='newline_delimited',
                            maximum_object_size=10485760) r
        INNER JOIN sampled_users s ON r.user_id = s.user_id
        WHERE r.timestamp >= 1577836800 AND r.timestamp <= 1703980800
    ) TO 'sample-duckdb/sampled_reviews_temporal.parquet' (FORMAT PARQUET)
""")

print("Done!")
con.close()